In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
high_risk_df = df.filter((df["Attrition"] == "No") & (df["JobSatisfaction"] < 3)) \
    .select("EmployeeNumber", "EmployeeCount", "Department", "JobRole", "JobSatisfaction", "Age", "Gender", "MaritalStatus", "OverTime", "MonthlyIncome", "YearsAtCompany", "Attrition")

high_risk_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history().select("version")
display(history_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
spark.sql("""
INSERT INTO `external-catalog`.default.high_risk_attrition_employees
(EmployeeNumber, EmployeeCount, Department, JobRole, JobSatisfaction, Age, Gender, MaritalStatus, OverTime, MonthlyIncome, YearsAtCompany, Attrition)
VALUES (99999, 1, 'DummyDept', 'DummyRole', 1, 30, 'M', 'Single', 'No', 1000, 1, 'No')
""")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("EmployeeNumber", IntegerType(), True),
    StructField("EmployeeCount", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("JobRole", StringType(), True),
    StructField("JobSatisfaction", IntegerType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("MaritalStatus", StringType(), True),
    StructField("OverTime", StringType(), True),
    StructField("MonthlyIncome", IntegerType(), True),
    StructField("YearsAtCompany", IntegerType(), True),
    StructField("Attrition", StringType(), True)
])

data = [
    (88888, 1, 'DummyDept2', 'DummyRole2', 2, 28, 'F', 'Married', 'Yes', 2000, 2, 'No')
]

dummy_df = spark.createDataFrame(data, schema)
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
history_df = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees").history().select("version")
display(history_df)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history().select("version", "timestamp", "operation")
display(history_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
df_v0 = spark.read.option('versionAsOf', 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v0)

df_v1 = spark.read.option('versionAsOf', 1).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v1)

In [0]:
df_timestamp = spark.read.option('timestampAsOf', '2026-02-21T20:50:10.316+00:00').table("`external-catalog`.default.high_risk_attrition_employees")
display(df_timestamp)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS `external-catalog`.default.transformed_data
""")

In [0]:
from pyspark.sql.functions import when, col

# Logical transformations: create a new column 'HighIncome' and flag employees with MonthlyIncome > 10000
transformed_df = df.withColumn(
    "HighIncome",
    when(col("MonthlyIncome") > 10000, "Yes").otherwise("No")
).withColumn(
    "IsYoung",
    when(col("Age") < 30, "Yes").otherwise("No")
)

# Write to volume, partitioned by Department
transformed_df.write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("Department") \
    .save("/Volumes/external-catalog/default/transformed_data/employee_transformed")